In [1]:
import numpy as np
from PositionalEncoding import *

np.random.seed(0)


# -----------------------
# vocab
# -----------------------

In [2]:
src_vocab = {
    "PAD":0,"BOS":1,"EOS":2,
    "one":3,"two":4,"three":5,"four":6
}

tgt_vocab = {
    "PAD":0,"BOS":1,"EOS":2,
    "uno":3,"dos":4,"tres":5,"cuatro":6
}

vocab_size = len(src_vocab)

# -----------------------
# lookup tables
# -----------------------

In [3]:
src_data = [
    [3,4,5],   # one two three
    [4,6],     # two four
    [6,3],     # four one
    [5,4]      # three two
]

tgt_data = [
    [3,4,5],
    [4,6],
    [6,3],
    [5,4]
]


## add BOS EOS

In [4]:
def prepare(seq):
    return [1] + seq + [2]


src_data = [prepare(s) for s in src_data]
tgt_data = [prepare(s) for s in tgt_data]

# -----------------------
# batching
# -----------------------

In [5]:
batch_size = 2

def get_batch(start_index):
    # Slice data
    src = src_data[start_index:start_index + batch_size]
    tgt = tgt_data[start_index:start_index + batch_size]

    # Actual batch size (may be < batch_size in last batch)
    B = len(src)

    max_src = max(len(x) for x in src)
    max_tgt = max(len(x) for x in tgt)

    src_batch = np.zeros((B, max_src), dtype=int)
    tgt_batch = np.zeros((B, max_tgt), dtype=int)

    for i, s in enumerate(src):
        src_batch[i, :len(s)] = s

    for i, t in enumerate(tgt):
        tgt_batch[i, :len(t)] = t

    return src_batch, tgt_batch


# -----------------------
# model
# -----------------------


In [6]:
d_model = 16
num_heads = 2

In [7]:
src_embed = Embedding(vocab_size,d_model)
tgt_embed = Embedding(vocab_size,d_model)

In [8]:
pos = PositionalEncoding(d_model)

In [9]:
encoder_attn = MultiHeadAttention(d_model,num_heads)

In [10]:
decoder_self = MultiHeadAttention(d_model, num_heads)
decoder_cross = MultiHeadAttention(d_model, num_heads)

In [11]:
output_layer = Layer_Dense(d_model,vocab_size)

# -----------------------
# training
# -----------------------

In [12]:
lr = 0.01
epochs = 300

for epoch in range(epochs):

    total_loss = 0

    for i in range(0, len(src_data), batch_size):

        src_batch, tgt_batch = get_batch(i)   # shapes: (B,T)

        # -------------------
        # FORWARD (full batch)
        # -------------------

        # ----- Encoder -----
        enc = src_embed.forward(src_batch)       # (B,T,D)
        enc = pos.forward(enc)
        enc_out = encoder_attn.forward(enc, enc, enc)

        # ----- Decoder -----
        dec_input = tgt_batch[:, :-1]            # (B,T-1)
        target    = tgt_batch[:, 1:]             # (B,T-1)

        dec = tgt_embed.forward(dec_input)
        dec = pos.forward(dec)

        mask = create_causal_mask(dec_input.shape[1])

        dec_self = decoder_self.forward(dec, dec, dec, mask)
        cross_out = decoder_cross.forward(dec_self, enc_out, enc_out)

        dec_out = dec_self + cross_out           # residual

        logits = output_layer.forward(dec_out)   # (B,T-1,V)

        # ------------ PAD MASK ------------
        # target: (B, T-1), PAD id = 0
        pad_mask = (target != 0).astype(float)  # 1 where not PAD, 0 where PAD

        # compute batch loss + gradient, ignoring PAD positions
        loss, dlogits = softmax_cross_entropy(logits, target, mask=pad_mask)
        total_loss += loss

        # -------------------
        # BACKWARD (full batch)
        # -------------------

        d_dec_out = output_layer.backward(dlogits)

        d_dec_self = d_dec_out
        d_cross_out = d_dec_out

        # cross-attention backward
        dQ_cross, dK_cross, dV_cross = decoder_cross.backward(d_cross_out)
        ddec = dQ_cross
        denc = dK_cross + dV_cross

        # decoder self-attention backward
        dQ_self, dK_self, dV_self = decoder_self.backward(d_dec_self)
        ddec += dQ_self + dK_self + dV_self

        # embedding backward
        tgt_embed.backward(ddec)

        # encoder backward
        denc_Q, denc_K, denc_V = encoder_attn.backward(denc)
        denc_total = denc_Q + denc_K + denc_V
        src_embed.backward(denc_total)

        # -------------------
        # UPDATE
        # -------------------
        output_layer.update(lr)
        decoder_self.update(lr)
        decoder_cross.update(lr)
        encoder_attn.update(lr)
        tgt_embed.update(lr)
        src_embed.update(lr)

    if epoch % 50 == 0:
        print(f"epoch {epoch}, loss {total_loss}")


epoch 0, loss 3.8911020357870627
epoch 50, loss 3.7589905743353302
epoch 100, loss 3.659321515292265
epoch 150, loss 3.583545716861541
epoch 200, loss 3.5252559679801267
epoch 250, loss 3.4797333709639306


# ------------------------

# Inference

# -------------------------

In [17]:
def translate(src_tokens, max_len=10):
    # ----- ENCODER -----
    src_tokens = np.array(src_tokens, dtype=int)[None, :]   # (1, T_src)

    enc = src_embed.forward(src_tokens)   # (1, T_src, D)
    enc = pos.forward(enc)               # (1, T_src, D)
    enc_out = encoder_attn.forward(enc, enc, enc)  # (1, T_src, D)

    # ----- DECODER -----
    BOS, EOS = 1, 2
    generated = [BOS]

    for _ in range(max_len):
        # keep decoder tokens batched: (1, T_dec)
        dec_input = np.array(generated, dtype=int)[None, :]   # (1, T_dec)

        dec = tgt_embed.forward(dec_input)    # (1, T_dec, D)
        dec = pos.forward(dec)               # (1, T_dec, D)

        mask = create_causal_mask(dec_input.shape[1])  # (1,1,T_dec,T_dec)

        # masked self-attention (Q=K=V=dec)
        dec_self = decoder_self.forward(dec, dec, dec, mask)    # (1, T_dec, D)

        # cross-attention (Q = decoder, K,V = encoder)
        cross_out = decoder_cross.forward(dec_self, enc_out, enc_out)  # (1, T_dec, D)

        # residual
        dec_out = dec_self + cross_out          # (1, T_dec, D)

        # output projection
        logits = output_layer.forward(dec_out)  # (1, T_dec, V)
        last_logits = logits[0, -1]             # (V,)
        probs = softmax(last_logits)            # (V,)
        next_token = int(np.argmax(probs))

        generated.append(next_token)

        if next_token == EOS:
            break

    return generated


In [18]:
src = [1, 3,4,5, 2]   # BOS one two three EOS
print(translate(src))


[1, 2]
